# EDA — YouToxic

- **Dropped:** `CommentId`, `VideoId` (not used anywhere in the project)
- **Model input:** `Text`
- **Model target:** `IsToxic` (Essential phase)
- **Kept for analysis / later levels:** all other `Is*` label columns

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.load_data import (
    TARGET_COLUMN,
    label_columns,
    load_dataset,
    prepare_xy,
    without_ids,
)

In [ ]:
raw = load_dataset(ROOT / "data/raw/youtoxic_english_1000.csv")
df = without_ids(raw)
df.head()

In [ ]:
print("Rows:", len(df))
print("Columns:", list(df.columns))
print("Label columns:", label_columns(df))
print("\nMissing Text:", df["Text"].isna().sum())
print("Empty Text:", (df["Text"].astype(str).str.strip() == "").sum())
print("Duplicate texts:", df["Text"].duplicated().sum())

In [ ]:
target_counts = df[TARGET_COLUMN].value_counts()
print(target_counts)
print(f"{TARGET_COLUMN} positive rate: {target_counts.get(True, 0) / len(df):.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
target_counts.plot(kind="bar", ax=ax, color=["#4c78a8", "#e45756"])
ax.set_title(f"{TARGET_COLUMN} distribution")
ax.set_xlabel(TARGET_COLUMN)
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
label_prev = {}
for col in label_columns(df):
    label_prev[col] = df[col].astype(bool).mean()

prev = pd.Series(label_prev).sort_values(ascending=False)
print(prev.apply(lambda x: f"{x:.1%}"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
prev.plot(kind="barh", ax=ax, color="#72b7b2")
ax.set_title("Label prevalence (all Is* columns)")
ax.set_xlabel("Share of comments")
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df[TARGET_COLUMN], df["IsAbusive"], margins=True)

In [ ]:
bool_labels = df[label_columns(df)].astype(bool)
co_occurrence = bool_labels.T.dot(bool_labels)
co_occurrence

In [ ]:
df["text_length"] = df["Text"].astype(str).str.len()
df.groupby(TARGET_COLUMN)["text_length"].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, subset in df.groupby(TARGET_COLUMN):
    ax.hist(subset["text_length"], bins=40, alpha=0.6, label=str(label))
ax.set_title(f"Comment length by {TARGET_COLUMN}")
ax.set_xlabel("Characters")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
x, y = prepare_xy(raw)
print(f"Training samples after prepare_xy: {len(x)}")
print(f"Dropped from raw: {len(df) - len(x)}")
print(f"{TARGET_COLUMN} positive rate: {y.mean():.2%}")

In [ ]:
for label, title in [(1, "Toxic"), (0, "Non-toxic")]:
    print(f"\n=== {title} (first 3) ===")
    for text in x[y == label].head(3):
        print(text[:280], "\n---")

## Takeaways

- **Essential model:** `Text` → `IsToxic` only; auxiliary `Is*` columns inform EDA and future levels (multi-label, error analysis).
- `IsAbusive` and other subtypes overlap with `IsToxic` — useful context, not training features (avoids label leakage).
- Slight class imbalance → `class_weight='balanced'` in training.
- Long comments / URLs → regex preprocessing matters.